In [7]:
import joblib

import pandas as pd
rf_model_loaded = joblib.load('delivery_model.pkl')
sc_model_loaded = joblib.load('delivery_scaler.pkl')




In [8]:
def predict_delivery_status(order_details, training_column):
    new_order_df = pd.DataFrame([order_details])
    new_order_encode = pd.get_dummies(new_order_df)
    new_order_alligned = new_order_encode.reindex(columns=training_column, fill_value=0)
    new_order_scaled = sc_model_loaded.transform(new_order_alligned)
    predict = rf_model_loaded.predict(new_order_scaled)
    return predict[0]


In [9]:
training_columns = ['order_value', 'distance_km', 'customer_rating_history', 'num_items',
    'promo_applied', 'order_month', 'order_of_the_week',
    'category_beauty', 'category_books', 'category_clothing', 'category_electronics',
    'category_groceries', 'category_home & kitchen',
    'warehouse_city_Chattogram', 'warehouse_city_Dhaka', 'warehouse_city_Khulna',
    'warehouse_city_Rajshahi', 'warehouse_city_Sylhet',
    'courier_partner_e-courier', 'courier_partner_pathao', 'courier_partner_redx',
    'courier_partner_sa paribahan', 'courier_partner_sundarban',
    'payment_method_Card', 'payment_method_Cash on Delivery', 'payment_method_Mobile Banking']

joblib.dump(training_columns, 'delivery_columns.pkl')

['delivery_columns.pkl']

In [10]:
new_order = {
    'order_value': 3200,
    'distance_km': 45,
    'customer_rating_history': 4.2,
    'num_items': 3,
    'promo_applied': 1,
    'order_month': 7,
    'order_of_the_week': 5,
    'category_electronics': 1,
    'warehouse_city_Dhaka': 1,
    'courier_partner_sundarban': 1,
    'payment_method_Cash on Delivery': 1
}

result = predict_delivery_status(new_order, training_columns)
print(result)

Cancelled


In [11]:
df.columns

NameError: name 'df' is not defined

In [1]:
import streamlit as st
import pandas as pd
import joblib

In [3]:
rf_model_loaded = joblib.load('delivery_model.pkl')
sc_model_loaded = joblib.load('delivery_scaler.pkl')
training_columns = joblib.load('delivery_columns.pkl')

In [ ]:
st.title('Delivery outcome predictor')
st.write("Enter a  new order details to find out if the order get delayed, on time, or cancelled.")

In [15]:
order_value = st.number_input("Order Value (taka)", min_value=100, max_value=30000, value=2000)
distance_km = st.number_input("Distance (km)", min_value=1, max_value=150, value=15)
customer_rating_history = st.slider("Customer Rating History", 1.0, 5.0, 4.0)
num_items = st.number_input("Number of Items", min_value=1, max_value=15, value=2)
promo_applied = st.selectbox("Promo Applied?", ["No", "Yes"])
order_month = st.selectbox("Order Month", list(range(1, 13)))
order_dayofweek = st.selectbox("Day of Week (0=Mon, 6=Sun)", list(range(0, 7)))

category = st.selectbox("Category", ["electronics", "clothing", "groceries", "home & kitchen", "beauty", "books"])
warehouse_city = st.selectbox("Warehouse City", ["Dhaka", "Chattogram", "Sylhet", "Khulna", "Rajshahi"])
courier_partner = st.selectbox("Courier Partner", ["pathao", "redx", "e-courier", "sa paribahan", "sundarban"])
payment_method = st.selectbox("Payment Method", ["Cash on Delivery", "Card", "Mobile Banking"])

# --- When the button is clicked, build the row and predict ---
if st.button("Predict Delivery Outcome"):

    # Step 1: build the raw order as a dictionary
    order_details = {
        'order_value': order_value,
        'distance_km': distance_km,
        'customer_rating_history': customer_rating_history,
        'num_items': num_items,
        'promo_applied': 1 if promo_applied == "Yes" else 0,
        'order_month': order_month,
        'order_of_the_week': order_dayofweek,
        f'category_{category}': 1,
        f'warehouse_city_{warehouse_city}': 1,
        f'courier_partner_{courier_partner}': 1,
        f'payment_method_{payment_method}': 1,
    }

    # Step 2: turn it into a dataframe, one-hot encode, align columns (same as before)
    new_order_df = pd.DataFrame([order_details])
    new_order_encoded = pd.get_dummies(new_order_df)
    new_order_aligned = new_order_encoded.reindex(columns=training_columns, fill_value=0)

    # Step 3: scale using the SAME scaler from training
    new_order_scaled = sc_model_loaded.transform(new_order_aligned)

    # Step 4: predict
    prediction = rf_model_loaded.predict(new_order_scaled)[0]
    probabilities = rf_model_loaded.predict_proba(new_order_scaled)[0]

    st.subheader(f"Prediction: {prediction}")

    # Show confidence for each class
    st.write("Confidence breakdown:")
    prob_df = pd.DataFrame({
        'Outcome': rf_model_loaded.classes_,
        'Probability': probabilities
    }).sort_values('Probability', ascending=False)
    st.dataframe(prob_df)

2026-09-08 15:14:41.025 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-08 15:14:41.028 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-08 15:14:41.029 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-08 15:14:41.031 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-08 15:14:41.032 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-08 15:14:41.033 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-08 15:14:41.036 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-09-08 15:14:41.039 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar